In [2]:
# !pip install langchain langchain-openai langchain-community langgraph python-dotenv

import os, json, time, textwrap, warnings
warnings.filterwarnings("ignore")              # keep teaching output clean

from dotenv import load_dotenv
import truststore; truststore.inject_into_ssl()   # plays nice with corporate TLS / proxies

# ── compatibility shim (safe to delete on Colab / clean machines) ───────────────
# On some setups `langchain-text-splitters` eagerly imports sentence-transformers,
# which imports torchcodec, which needs a specific ffmpeg build. We do no audio/video
# here, so if the real torchcodec can't load, drop in a harmless stub.
import sys, types, importlib.machinery
try:
    import torchcodec  # noqa: F401
except Exception:
    def _stub(name, **attrs):
        m = types.ModuleType(name)
        m.__spec__ = importlib.machinery.ModuleSpec(name, loader=None)
        for k, v in attrs.items():
            setattr(m, k, v)
        sys.modules[name] = m
        return m
    _tc = _stub("torchcodec")
    _tc.decoders = _stub("torchcodec.decoders", AudioDecoder=object, VideoDecoder=object)
    for _s in ("encoders", "samplers", "transforms"):
        setattr(_tc, _s, _stub("torchcodec." + _s))

# ── API key ─────────────────────────────────────────────────────────────────────
load_dotenv('/Users/shivam13juna/Documents/scaler/GEN_AI_REF/openai_key.env')
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY (e.g. in openai_key.env)"

MODEL = "gpt-5-nano"   # cheap + fast; everything here works on any chat model

def show(text, width=90):
    "Pretty-print a long string wrapped to the terminal width."
    print(textwrap.fill(str(text), width=width))

print("Setup OK - model:", MODEL)

Setup OK - model: gpt-5-nano


In [3]:
from openai import OpenAI
client = OpenAI()

blurb = "ANC over-ear BT5.3 headphones, 30h batt, foldable, multipoint."

resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You rewrite terse product specs into one friendly sentence."},
        {"role": "user",   "content": blurb},
    ],
)
print(resp.choices[0].message.content)

These ANC over-ear headphones with Bluetooth 5.3 offer up to 30 hours of battery life, fold for easy portability, and support multipoint pairing.


In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL, temperature=0)

ai_msg = llm.invoke([
    {"role": "system", "content": "You rewrite terse product specs into one friendly sentence."},
    {"role": "user",   "content": blurb},
])
print(ai_msg.content)

These ANC over-ear headphones with Bluetooth 5.3 offer up to 30 hours of battery life, fold for easy portability, and support multipoint pairing.


In [6]:
ai_msg = llm.invoke("What's IQ of donald trump?")
print(ai_msg.content)

There is no verified public IQ score for Donald Trump. He has sometimes claimed to have a high IQ, but he has not released an official score, and any numbers you see in the media are unverified or based on self-reports or rumors. So, there isn’t a credible, published IQ score to quote. If you’d like, I can summarize what media reports say and where they come from, or explain how IQ testing works and why such scores aren’t typically public for public figures.


- `.invoke(x)` — run once
- `.batch([x, y, z])` — run many, concurrently
- `.stream(x)` — stream the output as it's generated
- `.ainvoke` / `.abatch` / `.astream` — async versions of each

In [10]:
from langchain_core.prompts import ChatPromptTemplate

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are TechMart's copywriter. Rewrite the spec as {style} copy, one sentence."),
    ("human",  "{spec}"),
])

# Fill it in to see exactly what the model will receive:
for m in rewrite_prompt.invoke({"style": "playful", "spec": blurb}).messages:
    print(f"{m.type:6s}: {m.content}")

system: You are TechMart's copywriter. Rewrite the spec as playful copy, one sentence.
human : ANC over-ear BT5.3 headphones, 30h batt, foldable, multipoint.
